# Práca s rodinou modelov Meta

## Úvod

Táto lekcia pokryje:

- Preskúmanie dvoch hlavných modelov rodiny Meta - Llama 3.1 a Llama 3.2, plus aktuálne dostupných alternatív použitých v ukážkach kódu (Llama 3.3 a Llama 4)
- Pochopenie použití a scenárov pre každý model
- Ukážka kódu na zobrazenie jedinečných vlastností každého modelu


## Rodina modelov Meta

V tejto lekcii preskúmame 2 modely z rodiny Meta alebo "Llama Herd" - Llama 3.1 a Llama 3.2. Spustiteľné ukážky kódu používajú Llamu 3.3 a Llamu 4, ktoré sú aktuálne dostupné alternatívy v katalógu Microsoft Foundry Models.

Tieto modely sú dostupné v rôznych variantoch a nájdete ich v [Microsoft Foundry Models katalógu](https://ai.azure.com/catalog/models?WT.mc_id=academic-105485-koreyst).

> **Poznámka:** GitHub Models končí na konci júla 2026. Tu sú ďalšie podrobnosti o používaní [Microsoft Foundry Models](https://learn.microsoft.com/azure/ai-foundry/model-inference/overview?WT.mc_id=academic-105485-koreyst) na prototypovanie s AI modelmi.

> **Poznámka:** Konkrétne verzie modelov, pre ktoré bola táto lekcia pôvodne napísaná (Llama 3.1 70B/405B Instruct, Llama 3.2 11B/90B Vision Instruct), boli následne vyradené z katalógu Microsoft Foundry Models. Nižšie uvedené ukážky kódu používajú `Llama-3.3-70B-Instruct` a `Llama-4-Scout-17B-16E-Instruct` ako aktuálne dostupné alternatívy v katalógu Microsoft Foundry Models, takže stále bežia.

Varianty modelov (ako pôvodne uvedené v tejto lekcii):
- Llama 3.1 - 70B Instruct (vyradený)
- Llama 3.1 - 405B Instruct (vyradený)
- Llama 3.2 - 11B Vision Instruct (vyradený)
- Llama 3.2 - 90B Vision Instruct (vyradený)

Aktuálne dostupné alternatívy použité v ukážkach kódu:
- Llama 3.3 - 70B Instruct (`Llama-3.3-70B-Instruct`)
- Llama 4 Scout - 17B 16E Instruct (`Llama-4-Scout-17B-16E-Instruct`)

*Poznámka: Vyššie diskutované varianty Llama 3 sú uvedené len pre historický kontext a už nie sú dostupné v Microsoft Foundry Models.*



## Llama 3.1 

Pri 405 miliardách parametrov spadá Llama 3.1 do kategórie open source LLM. 

Tento model je vylepšením oproti skoršej verzii Llama 3 a ponúka: 

- Väčšie kontextové okno - 128k tokenov oproti 8k tokenom 
- Väčší maximálny počet výstupných tokenov - 4096 oproti 2048 
- Lepšia podpora viacerých jazykov - vďaka zvýšeniu počtu tréningových tokenov 

To umožňuje Llama 3.1 zvládať komplexnejšie prípady použitia pri tvorbe aplikácií GenAI vrátane: 
- Nativne volanie funkcií - schopnosť volať externé nástroje a funkcie mimo LLM workflow
- Lepší výkon RAG - vďaka väčšiemu kontextovému oknu 
- Generovanie syntetických dát - schopnosť vytvárať efektívne dáta pre úlohy ako doladenie 



### Volanie natívnych funkcií 

Llama 3.1 bola doladená tak, aby bola efektívnejšia pri vykonávaní volaní funkcií alebo nástrojov. Má tiež dva vstavané nástroje, ktoré model môže identifikovať ako potrebné použiť na základe promptu od používateľa. Tieto nástroje sú: 

- **Brave Search** - Môže byť použitý na získanie aktuálnych informácií, ako je počasie, pomocou vyhľadávania na webe 
- **Wolfram Alpha** - Môže byť použitý na zložitejšie matematické výpočty, takže nie je potrebné písať vlastné funkcie. 

Môžete si tiež vytvoriť vlastné nástroje, ktoré môže LLM volať. 

V nasledujúcom príklade kódu: 

- Definujeme dostupné nástroje (brave_search, wolfram_alpha) v systémovom promte. 
- Odosielame užívateľský prompt, ktorý sa pýta na počasie v určitom meste. 
- LLM odpovie volaním nástroja Brave Search, ktoré bude vyzerať takto `<|python_tag|>brave_search.call(query="Stockholm weather")` 

*Poznámka: Tento príklad iba vykoná volanie nástroja, ak chcete získať výsledky, budete si musieť vytvoriť bezplatný účet na stránke Brave API a definovať samotnú funkciu` 


In [ ]:
%pip install azure-core
%pip install azure-ai-inference

In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import AssistantMessage, SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]
# Meta-Llama-3.1-405B-Instruct was retired from Microsoft Foundry on 2026-06-13 with no replacement listed;
# using Llama-3.3-70B-Instruct, a currently available alternative, instead
model_name = "Llama-3.3-70B-Instruct"

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


tool_prompt=f"""
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Environment: ipython
Tools: brave_search, wolfram_alpha
Cutting Knowledge Date: December 2023
Today Date: 23 July 2024

You are a helpful assistant<|eot_id|>
"""

messages = [
    SystemMessage(content=tool_prompt),
    UserMessage(content="What is the weather in Stockholm?"),

]

response = client.complete(messages=messages, model=model_name)

print(response.choices[0].message.content)






### Llama 3.2 

Napriek tomu, že je LLM, jedným z obmedzení Llama 3.1 je multimodalita. To znamená schopnosť používať rôzne typy vstupov, ako sú obrázky ako podnety a poskytovať na ne odpovede. Táto schopnosť je jednou z hlavných vlastností Llama 3.2. Tieto vlastnosti zahŕňajú aj: 

- Multimodalita - schopnosť vyhodnocovať textové aj obrazové podnety 
- Variácie od malých po stredné veľkosti (11B a 90B) - to poskytuje flexibilné možnosti nasadenia, 
- Iba textové variácie (1B a 3B) - umožňuje nasadiť model na edge / mobilných zariadeniach a poskytuje nízku latenciu 

Podpora multimodality predstavuje veľký krok vo svete open source modelov. Nižšie uvedený príklad kódu využíva ako obraz, tak textový podnet na získanie analýzy obrázka. Pretože Llama 3.2 90B Vision Instruct bol stiahnutý, príklad používa `Llama-4-Scout-17B-16E-Instruct`, aktuálne dostupnú natívne multimodálnu alternatívu. 

### Podpora multimodality s Llama 3.2


In [ ]:
%pip install azure-core
%pip install azure-ai-inference

In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import (
    SystemMessage,
    UserMessage,
    TextContentItem,
    ImageContentItem,
    ImageUrl,
    ImageDetailLevel,
)
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]
# Llama-3.2-90B-Vision-Instruct was retired from Microsoft Foundry on 2026-06-13 with no replacement listed;
# using Llama-4-Scout-17B-16E-Instruct, a currently available natively multimodal alternative, instead
model_name = "Llama-4-Scout-17B-16E-Instruct"


client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)

response = client.complete(
    messages=[
        SystemMessage(
            content="You are a helpful assistant that describes images in details."
        ),
        UserMessage(
            content=[
                TextContentItem(text="What's in this image?"),
                ImageContentItem(
                    image_url=ImageUrl.load(
                        image_file="sample.jpg",
                        image_format="jpg",
                        detail=ImageDetailLevel.LOW)
                ),
            ],
        ),
    ],
    model=model_name,
)

print(response.choices[0].message.content)


## Učenie tu nekončí, pokračujte v ceste

Po dokončení tejto lekcie si pozrite našu [kolekciu učenia Generatívnej AI](https://aka.ms/genai-collection?WT.mc_id=academic-105485-koreyst), aby ste pokračovali v zlepšovaní svojich vedomostí o Generatívnej AI!


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Vyhlásenie o zodpovednosti**:
Tento dokument bol preložený pomocou AI prekladateľskej služby [Co-op Translator](https://github.com/Azure/co-op-translator). Hoci sa snažíme o presnosť, vezmite prosím na vedomie, že automatické preklady môžu obsahovať chyby alebo nepresnosti. Pôvodný dokument v jeho natívnom jazyku by mal byť považovaný za autoritatívny zdroj. Pre kritické informácie sa odporúča profesionálny ľudský preklad. Nie sme zodpovední za žiadne nedorozumenia alebo nesprávne interpretácie vyplývajúce z použitia tohto prekladu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
